# F1 – Definición del proyecto
## Análisis de patrones temporales de generación eléctrica
### TER CMPC Laja, TER CMPC Pacífico y TER CMPC Santa Fe

**Fuente:** Generación Real – Coordinador Eléctrico Nacional (CEN)  
**Periodo:** enero–agosto de 2026

**Integrantes**: Verónica Durán Cisterna; Raúl Moya Arriagada; Daniela Rojas Vilches; Manuel Sánchez Cárcamo 
**Asignatura**: Programación para la Ciencia de Datos

Este notebook corresponde a la **Fase 1 (F1)**. Define la problemática, pregunta de investigación, objetivos, alcance, supuestos y configuración reproducible inicial. La transformación completa se desarrolla en F2.

## 1. Problemática

El CEN publica información de Generación Real que permite estudiar el comportamiento temporal de distintas centrales. La fuente utilizada contiene **359.891 registros y 33 variables** para enero–agosto de 2026 y está estructurada en formato ancho: cada registro representa una central en una fecha, con mediciones entre `Hora 1` y `Hora 24`.

El proyecto se concentra en `TER CMPC LAJA`, `TER CMPC PACIFICO` y `TER CMPC SANTA FE`. Para analizarlas de manera consistente es necesario definir una granularidad única y preparar posteriormente una estructura horaria validada.

La problemática consiste en **comprender cómo se comporta temporalmente la generación eléctrica reportada para estas tres centrales**, mediante un proceso reproducible y trazable.

La unidad de observación del dataset analítico queda definida como una combinación central–fecha–hora, con la generación eléctrica reportada en MWh.

## 2. Investigación

> **¿Qué patrones temporales de generación eléctrica caracterizan a las centrales TER CMPC Laja, TER CMPC Pacífico y TER CMPC Santa Fe durante el período enero–agosto de 2026?**

Esta pregunta es el eje del proyecto. La carga, transformación, validación, documentación y control de versiones constituyen la metodología para construir la información necesaria para responderla.

## 3. Objetivo general

**Analizar los patrones temporales de generación eléctrica de las centrales TER CMPC Laja, TER CMPC Pacífico y TER CMPC Santa Fe durante enero–agosto de 2026, a partir de los datos públicos de Generación Real del Coordinador Eléctrico Nacional, mediante un proceso reproducible y trazable de preparación, validación y análisis de datos.**

## 4. Objetivos específicos

1. **Caracterizar** la distribución de la generación eléctrica de cada central considerando su comportamiento horario, diario y mensual.
2. **Comparar** los patrones temporales de generación entre las tres centrales.
3. **Caracterizar** la frecuencia y distribución temporal de los registros de generación igual a `0 MWh`, sin atribuirles una causa operacional no respaldada por la fuente.
4. **Identificar** regularidades y diferencias temporales relevantes que sirvan de base para las etapas posteriores de análisis e interpretación.

Las operaciones de carga, transformación, validación y versionamiento se consideran parte de la metodología técnica y no objetivos de investigación.

## 5. Fuente, alcance y unidad de observación

**Fuente pública:** Generación Real – Coordinador Eléctrico Nacional (CEN)  
https://www.coordinador.cl/operacion/graficos/operacion-real/generacion-real-/

**Diccionario:**  
https://www.coordinador.cl/wp-content/uploads/2022/11/B43-DIN-04-Diccionario-de-Datos-Web-SIP.pdf

- Fuente original: **359.891 × 33**.
- Periodo: **enero–agosto de 2026**.
- Alcance: **3 centrales**.
- Subconjunto previo a transformación: **729 registros** (243 fechas por central).
- Resultado esperado F2: **17.496 observaciones × 13 variables**.

**Unidad de observación analítica:** una central en una fecha y una hora determinada, con su generación reportada en MWh.

La fuente original se conserva sin modificaciones en `data/raw/`. Los datos se utilizan con fines académicos y con atribución al CEN; no se atribuye una licencia abierta específica que la institución no haya declarado expresamente.

## 6. Variables y roles analíticos

| Variable | Rol |
|---|---|
| `Generacion_MWh` | Numérica continua; variable principal |
| `Hora` | Numérica discreta; dimensión horaria |
| `Fecha` | Fecha; dimensión temporal |
| `Central` | Categórica nominal |
| `Dia_Semana` | Categórica ordinal derivada |
| `ID_Observacion` | Identificador único |

`Coordinado`, `Tipo` y `Subtipo` se conservan como variables descriptivas. Se comprobará programáticamente su número de valores únicos en el subconjunto para declarar si presentan varianza cero.

## 7. Supuestos y decisiones metodológicas

- El alcance se limita a las tres centrales declaradas.
- La granularidad analítica será **central–fecha–hora**.
- Los valores `0 MWh` se conservan como observaciones válidas reportadas por la fuente.
- **No se atribuye automáticamente una causa operacional a los ceros**, porque la fuente no aporta evidencia suficiente para afirmar detención, falla u otra causa específica.
- F1 define e inspecciona; F2 transforma y valida; F3 profundizará el análisis y F4 integrará e interpretará los resultados.

## 8. Reproducibilidad y trazabilidad

**Reproducibilidad:** capacidad de volver a ejecutar el flujo desde la fuente mediante el entorno, dependencias e instrucciones declaradas. Se apoya en `.venv`, `requirements.txt`, rutas reproducibles, notebooks y validaciones programáticas.

**Trazabilidad:** capacidad de identificar qué cambió, quién realizó una modificación y cómo evolucionó el proyecto. Se apoya en Git, GitHub, commits descriptivos, ramas, Pull Requests, documentación y evidencias.

Son conceptos relacionados, pero distintos.

## 9. Documentación científica y fases

La documentación científica incluye celdas narrativas Markdown, descripción/diccionario de variables, registro de decisiones, README, referencias y evidencias.

**F1 – Definición:** pregunta, objetivos, alcance, supuestos, fuente, unidad de observación y configuración inicial.  
**F2 – Preparación:** carga, exploración, selección, transformación ancho–largo, validación y exportación.  
**F3 – Análisis:** patrones temporales, comparación entre centrales y caracterización de ceros.  
**F4 – Integración:** interpretación, conclusiones y comunicación.

F1 y F2 corresponden al avance materializado; F3 y F4 quedan proyectadas.

## 10. Configuración inicial del entorno

In [7]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.14.3
pandas: 3.0.5
NumPy: 2.5.3
Matplotlib: 3.11.1


## 11. Ruta reproducible

La función evita depender de una ruta absoluta de un computador específico.

In [8]:
def encontrar_raiz_proyecto(inicio=None):
    """Localiza la raíz identificando README.md y las carpetas F1 y F2."""
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if ((candidata / "README.md").exists()
            and (candidata / "F1").exists()
            and (candidata / "F2").exists()):
            return candidata
    raise FileNotFoundError("No fue posible localizar la raíz del proyecto.")

RAIZ = encontrar_raiz_proyecto()
RUTA_RAW = RAIZ / "data" / "raw" / "generacion_real_cen_ene_ago_2026.csv"
print("Raíz:", RAIZ)
print("Fuente:", RUTA_RAW)

Raíz: C:\Users\eliza\Desktop\ESTUDIOS\MAGÍSTER\1 PROGRAMACIÓN PARA LA CIENCIA\Proyecto
Fuente: C:\Users\eliza\Desktop\ESTUDIOS\MAGÍSTER\1 PROGRAMACIÓN PARA LA CIENCIA\Proyecto\data\raw\generacion_real_cen_ene_ago_2026.csv


## 12. Carga y validación inicial de la fuente

In [9]:
if not RUTA_RAW.exists():
    raise FileNotFoundError(f"No se encontró la fuente pública en: {RUTA_RAW}")

df_raw = pd.read_csv(RUTA_RAW, sep=";", decimal=",", encoding="utf-8-sig")

columnas_base = ["Año","Mes","Llave","Central","Coordinado","Grupo reporte","Tipo","Subtipo","Fecha"]
columnas_hora = [f"Hora {i}" for i in range(1, 25)]
faltantes = [c for c in columnas_base + columnas_hora if c not in df_raw.columns]

assert not faltantes, f"Faltan columnas: {faltantes}"
assert df_raw.shape[0] >= 2000
assert df_raw.shape[1] >= 12

print(f"Fuente: {len(df_raw):,} registros × {df_raw.shape[1]} variables")
print("Duplicados exactos:", int(df_raw.duplicated().sum()))
print("Nulos:", int(df_raw.isna().sum().sum()))
display(df_raw.head())

Fuente: 359,891 registros × 33 variables
Duplicados exactos: 0
Nulos: 0


,Año,Mes,Llave,Central,Coordinado,Grupo reporte,Tipo,Subtipo,Fecha,Hora 1,...,Hora 15,Hora 16,Hora 17,Hora 18,Hora 19,Hora 20,Hora 21,Hora 22,Hora 23,Hora 24
0,2026,Ene,BESS ALFALFAL VR1 (Inyección),BESS ALFALFAL VR1,-,BESS ALFALFAL VR1,BESS,Inyección,2026-01-01,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026,Ene,BESS ALFALFAL VR1 (Inyección),BESS ALFALFAL VR1,-,BESS ALFALFAL VR1,BESS,Inyección,2026-01-02,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026,Ene,BESS ALFALFAL VR1 (Inyección),BESS ALFALFAL VR1,-,BESS ALFALFAL VR1,BESS,Inyección,2026-01-03,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026,Ene,BESS ALFALFAL VR1 (Inyección),BESS ALFALFAL VR1,-,BESS ALFALFAL VR1,BESS,Inyección,2026-01-04,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026,Ene,BESS ALFALFAL VR1 (Inyección),BESS ALFALFAL VR1,-,BESS ALFALFAL VR1,BESS,Inyección,2026-01-05,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 13. Delimitación del alcance

In [10]:
CENTRALES = ["TER CMPC LAJA", "TER CMPC PACIFICO", "TER CMPC SANTA FE"]

df_alcance = df_raw[df_raw["Central"].isin(CENTRALES)].copy()

resumen = (df_alcance.groupby("Central")
           .agg(registros=("Central","size"), fechas=("Fecha","nunique"))
           .reset_index())
display(resumen)

assert set(df_alcance["Central"].unique()) == set(CENTRALES)
assert len(df_alcance) == 729
assert df_alcance["Central"].nunique() == 3

,Central,registros,fechas
0,TER CMPC LAJA,243,243
1,TER CMPC PACIFICO,243,243
2,TER CMPC SANTA FE,243,243


## 14. Verificación de variables descriptivas

Se verifica expresamente si `Coordinado`, `Tipo` y `Subtipo` presentan un único valor dentro del alcance. Un único valor implica ausencia de capacidad discriminante dentro de este subconjunto.

In [11]:
variables = ["Coordinado", "Tipo", "Subtipo"]
revision = pd.DataFrame({
    "Variable": variables,
    "Valores_unicos": [df_alcance[c].nunique(dropna=False) for c in variables],
    "Valores": [", ".join(map(str, df_alcance[c].drop_duplicates().tolist())) for c in variables]
})
revision["Varianza_cero_en_alcance"] = revision["Valores_unicos"].eq(1)
display(revision)

,Variable,Valores_unicos,Valores,Varianza_cero_en_alcance
0,Coordinado,1,BIOENERGÍAS FORESTALES SPA,True
1,Tipo,1,Termoeléctricas,True
2,Subtipo,1,Biomasa,True


## 15. Resultado esperado de F2

F2 transforma las 24 mediciones horarias de los **729 registros** desde formato ancho a largo. Se esperan **17.496 observaciones × 13 variables**, con unidad central–fecha–hora y validaciones de nulos, duplicados, valores negativos y consistencia temporal.

In [12]:
OBSERVACIONES_ESPERADAS_F2 = len(df_alcance) * 24
assert OBSERVACIONES_ESPERADAS_F2 == 17496

print("VALIDACIÓN INICIAL COMPLETADA")
print(f"Registros fuente: {len(df_raw):,}")
print(f"Variables fuente: {df_raw.shape[1]}")
print(f"Registros del alcance: {len(df_alcance):,}")
print(f"Centrales del alcance: {df_alcance['Central'].nunique()}")
print(f"Observaciones esperadas F2: {OBSERVACIONES_ESPERADAS_F2:,}")

VALIDACIÓN INICIAL COMPLETADA
Registros fuente: 359,891
Variables fuente: 33
Registros del alcance: 729
Centrales del alcance: 3
Observaciones esperadas F2: 17,496


## 16. Registro de decisiones

| Decisión | Justificación |
|---|---|
| Fuente pública CEN | Favorece acceso y reproducibilidad |
| Tres centrales | Delimita el objeto de estudio |
| Formato ancho → largo | Define granularidad central–fecha–hora |
| Conservar 0 MWh | Son valores reportados; no se clasifican automáticamente como error |
| No atribuir causa a los ceros | Evita inferencias no respaldadas |
| Separar reproducibilidad/trazabilidad | Son propiedades metodológicas diferentes |
| F1 define; F2 transforma | Mantiene separación coherente entre fases |

## 17. Referencias iniciales

- Coordinador Eléctrico Nacional. *Generación Real*. https://www.coordinador.cl/operacion/graficos/operacion-real/generacion-real-/
- Coordinador Eléctrico Nacional. (2022). *Diccionario de Datos Web SIP*. https://www.coordinador.cl/wp-content/uploads/2022/11/B43-DIN-04-Diccionario-de-Datos-Web-SIP.pdf
- Material didáctico del curso – Semana 1. https://github.com/magistercienciadatos/Material_Didactico_Semana1

La bibliografía definitiva se completará en el informe conforme a los requisitos de fuentes docentes, técnicas y académicas.